In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:

!ls "/content/drive/MyDrive/Dataset"

PlantDoc-Dataset-master.zip


In [6]:
!unzip "/content/drive/MyDrive/Dataset/PlantDoc-Dataset-master.zip"

Archive:  /content/drive/MyDrive/Dataset/PlantDoc-Dataset-master.zip
5467f6012d78d1c446145d5f582da6096f852ae8
   creating: PlantDoc-Dataset-master/
  inflating: PlantDoc-Dataset-master/LICENSE.txt  
  inflating: PlantDoc-Dataset-master/PlantDoc_Examples.png  
  inflating: PlantDoc-Dataset-master/README.md  
   creating: PlantDoc-Dataset-master/test/
   creating: PlantDoc-Dataset-master/test/Apple Scab Leaf/
  inflating: PlantDoc-Dataset-master/test/Apple Scab Leaf/052609%20Hartman%20Crabapple%20scab%20single%20leaf.JPG.jpg  
  inflating: PlantDoc-Dataset-master/test/Apple Scab Leaf/1b321015-6e33-4f18-aade-888f4383fe92.jpeg.jpg  
  inflating: PlantDoc-Dataset-master/test/Apple Scab Leaf/28-500x375.jpg  
  inflating: PlantDoc-Dataset-master/test/Apple Scab Leaf/816.jpg  
  inflating: PlantDoc-Dataset-master/test/Apple Scab Leaf/apple%20scab%20leaf.jpg  
  inflating: PlantDoc-Dataset-master/test/Apple Scab Leaf/apple%20scabnew.jpg  
  inflating: PlantDoc-Dataset-master/test/Apple Scab Lea

In [7]:
import os

print(os.listdir())

['.config', 'drive', 'PlantDoc-Dataset-master', 'sample_data']


In [8]:
!ls "PlantDoc-Dataset-master"

LICENSE.txt  PlantDoc_Examples.png  README.md  test  train


In [9]:
from datasets import load_dataset

dataset = load_dataset(
    "imagefolder",
    data_dir="PlantDoc-Dataset-master"
)

print(dataset)

Resolving data files:   0%|          | 0/2342 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/236 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 2342
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 236
    })
})


In [10]:
from transformers import MobileNetV2ForImageClassification
from torchvision import transforms

model = MobileNetV2ForImageClassification.from_pretrained(
    "linkanjarad/mobilenet_v2_1.0_224-plant-disease-identification",
    ignore_mismatched_sizes=True
)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Model loaded.


In [19]:
# ==========================================
# LABELS
# ==========================================
label_names = dataset["train"].features["label"].names
num_classes = len(label_names)

print("Classes:", num_classes)

# ==========================================
# FIXED TRANSFORMS
# ==========================================
def transform_examples(example_batch):

    images = [
        image.convert("RGB")
        for image in example_batch["image"]
    ]

    pixel_values = [
        transform(image)
        for image in images
    ]

    return {
        "pixel_values": pixel_values,
        "labels": example_batch["label"]
    }

dataset["train"] = dataset["train"].with_transform(transform_examples)
dataset["test"] = dataset["test"].with_transform(transform_examples)

# ==========================================
# COLLATE FUNCTION
# ==========================================
def collate_fn(batch):

    pixel_values = torch.stack([
        x["pixel_values"]
        for x in batch
    ])

    labels = torch.tensor([
        x["labels"]
        for x in batch
    ])

    return {
        "pixel_values": pixel_values,
        "labels": labels
    }

print("Dataset formatting complete.")

Classes: 28
Dataset formatting complete.


In [20]:
from torch.utils.data import DataLoader, random_split

# ==========================================
# TRAIN / VALIDATION SPLIT
# ==========================================
train_size = int(0.8 * len(dataset["train"]))
val_size = len(dataset["train"]) - train_size

train_dataset, val_dataset = random_split(
    dataset["train"],
    [train_size, val_size]
)

print(f"Train Size: {len(train_dataset)}")
print(f"Validation Size: {len(val_dataset)}")

# ==========================================
# DATALOADERS
# ==========================================
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    dataset["test"],
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

print("Dataloaders ready.")

Train Size: 1873
Validation Size: 469
Dataloaders ready.


In [21]:
import torch
import torch.optim as optim
from transformers import get_linear_schedule_with_warmup

# ==========================================
# FREEZE BACKBONE
# ==========================================
for param in model.mobilenet_v2.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

print(f"Trainable tensors: {len(trainable_params)}")

# ==========================================
# FIX BATCHNORM
# ==========================================
def set_frozen_bn_eval(module):
    for m in module.modules():
        if isinstance(m, nn.BatchNorm2d):
            if not any(p.requires_grad for p in m.parameters()):
                m.eval()

# ==========================================
# OPTIMIZER
# ==========================================
optimizer = optim.AdamW(
    trainable_params,
    lr=1e-4,
    betas=(0.9, 0.999),
    weight_decay=1e-2
)

criterion = nn.CrossEntropyLoss()

EPOCHS_PHASE_1 = 10

total_steps = len(train_loader) * EPOCHS_PHASE_1

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

print("Phase 1 setup complete.")

Trainable tensors: 6
Phase 1 setup complete.


In [14]:
import torch
import torch.optim as optim
from transformers import get_linear_schedule_with_warmup

# ==========================================
# FREEZE BACKBONE
# ==========================================
for param in model.mobilenet_v2.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

print(f"Trainable tensors: {len(trainable_params)}")

# ==========================================
# FIX BATCHNORM
# ==========================================
def set_frozen_bn_eval(module):
    for m in module.modules():
        if isinstance(m, nn.BatchNorm2d):
            if not any(p.requires_grad for p in m.parameters()):
                m.eval()

# ==========================================
# OPTIMIZER
# ==========================================
optimizer = optim.AdamW(
    trainable_params,
    lr=1e-4,
    betas=(0.9, 0.999),
    weight_decay=1e-2
)

criterion = nn.CrossEntropyLoss()

EPOCHS_PHASE_1 = 10

total_steps = len(train_loader) * EPOCHS_PHASE_1

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

print("Phase 1 setup complete.")

Trainable tensors: 6
Phase 1 setup complete.


In [16]:
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import wandb

# ==========================================
# WANDB
# ==========================================
wandb.login()

if wandb.run is not None:
    wandb.finish()

run = wandb.init(
    project='plant-disease-mobilenet',
    name='Phase-1-FrozenBackbone-BNFixed-AdamW',
    config={
        'learning_rate': 1e-4,
        'epochs': 10,
        'batch_size': 32,
        'phase': '1_frozen_backbone_bn_eval',
        'num_classes': num_classes,
        'architecture': 'MobileNetV2_Linear_Head',
        'optimizer': 'AdamW'
    }
)

# ==========================================
# DEVICE
# ==========================================
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)

print(device)

# ==========================================
# TRAIN FUNCTION
# ==========================================
def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    criterion,
    device,
    epoch,
    total_epochs
):
    model.train()
    set_frozen_bn_eval(model)

    total_loss = 0
    all_preds = []
    all_labels = []

    progress_bar = tqdm(
        loader,
        desc=f"Epoch {epoch+1}/{total_epochs} [Train]"
    )

    for batch in progress_bar:

        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(pixel_values).logits

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            trainable_params,
            max_norm=1.0
        )

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

        progress_bar.set_postfix({
            'loss': loss.item()
        })

        wandb.log({
            'batch_loss': loss.item()
        })

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    return avg_loss, acc

# ==========================================
# VALIDATION FUNCTION
# ==========================================
def validate(
    model,
    loader,
    criterion,
    device,
    epoch,
    total_epochs
):
    model.eval()

    total_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for batch in tqdm(
            loader,
            desc=f"Epoch {epoch+1}/{total_epochs} [Val]"
        ):

            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(pixel_values).logits

            loss = criterion(outputs, labels)

            total_loss += loss.item()

            preds = torch.argmax(
                outputs,
                dim=1
            ).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    _, _, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average='weighted',
        zero_division=0
    )

    return avg_loss, acc, f1

print("Training functions ready.")

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


cuda
Training functions ready.


In [22]:
# ==========================================
# RUN PHASE 1
# ==========================================
print("\nStarting Phase 1 Training...")

for epoch in range(EPOCHS_PHASE_1):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        criterion,
        device,
        epoch,
        EPOCHS_PHASE_1
    )

    val_loss, val_acc, val_f1 = validate(
        model,
        val_loader,
        criterion,
        device,
        epoch,
        EPOCHS_PHASE_1
    )

    print(
        f"Epoch {epoch+1}: "
        f"Train Loss={train_loss:.4f}, "
        f"Train Acc={train_acc:.4f}, "
        f"Val Acc={val_acc:.4f}, "
        f"Val F1={val_f1:.4f}"
    )

    wandb.log({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_accuracy': train_acc,
        'val_loss': val_loss,
        'val_accuracy': val_acc,
        'val_f1_weighted': val_f1,
    })

print("Phase 1 Complete.")


Starting Phase 1 Training...


Epoch 1/10 [Train]:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 1/10 [Val]:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 1: Train Loss=3.2979, Train Acc=0.0806, Val Acc=0.1237, Val F1=0.0457


Epoch 2/10 [Train]:   0%|          | 0/59 [00:00<?, ?it/s]

...

Epoch 9/10 [Val]:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 9: Train Loss=2.4544, Train Acc=0.2766, Val Acc=0.2878, Val F1=0.2321


Epoch 10/10 [Train]:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 10/10 [Val]:   0%|          | 0/15 [00:00<?, ?it/s]

Epoch 10: Train Loss=2.4591, Train Acc=0.2653, Val Acc=0.2878, Val F1=0.2321
Phase 1 Complete.


In [23]:
# ==========================================
# PHASE 2 - PROGRESSIVE UNFREEZING
# ==========================================

for param in model.mobilenet_v2.parameters():
    param.requires_grad = False

# Unfreeze deeper layers
for param in model.mobilenet_v2.layer[-4:].parameters():
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

print(f"Trainable tensors after unfreezing: {len(trainable_params)}")

# ==========================================
# LOWER LEARNING RATE
# ==========================================
optimizer = optim.AdamW(
    trainable_params,
    lr=1e-5,
    betas=(0.9, 0.999),
    weight_decay=1e-2
)

EPOCHS_PHASE_2 = 5

total_steps = len(train_loader) * EPOCHS_PHASE_2

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

print("Phase 2 setup complete.")

Trainable tensors after unfreezing: 42
Phase 2 setup complete.


In [24]:
# ==========================================
# RUN PHASE 2
# ==========================================
print("\nStarting Phase 2 Fine-Tuning...")

for epoch in range(EPOCHS_PHASE_2):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        criterion,
        device,
        epoch,
        EPOCHS_PHASE_2
    )

    val_loss, val_acc, val_f1 = validate(
        model,
        val_loader,
        criterion,
        device,
        epoch,
        EPOCHS_PHASE_2
    )

    print(
        f"Phase 2 Epoch {epoch+1}: "
        f"Train Loss={train_loss:.4f}, "
        f"Train Acc={train_acc:.4f}, "
        f"Val Loss={val_loss:.4f}, "
        f"Val Acc={val_acc:.4f}, "
        f"Val F1={val_f1:.4f}"
    )

    wandb.log({
        'phase2_epoch': epoch + 1,
        'phase2_train_loss': train_loss,
        'phase2_train_accuracy': train_acc,
        'phase2_val_loss': val_loss,
        'phase2_val_accuracy': val_acc,
        'phase2_val_f1_weighted': val_f1,
    })

print("Phase 2 Complete.")


Starting Phase 2 Fine-Tuning...


Epoch 1/5 [Train]:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 1/5 [Val]:   0%|          | 0/15 [00:00<?, ?it/s]

Phase 2 Epoch 1: Train Loss=2.5856, Train Acc=0.2451, Val Loss=2.5910, Val Acc=0.2495, Val F1=0.1895


Epoch 2/5 [Train]:   0%|          | 0/59 [00:00<?, ?it/s]

...

Epoch 4/5 [Val]:   0%|          | 0/15 [00:00<?, ?it/s]

Phase 2 Epoch 4: Train Loss=2.4147, Train Acc=0.2782, Val Loss=2.4884, Val Acc=0.2431, Val F1=0.1949


Epoch 5/5 [Train]:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 5/5 [Val]:   0%|          | 0/15 [00:00<?, ?it/s]

Phase 2 Epoch 5: Train Loss=2.3913, Train Acc=0.2867, Val Loss=2.3872, Val Acc=0.2921, Val F1=0.2322
Phase 2 Complete.


In [25]:
# ==========================================
# CONTINUE PHASE 2
# ==========================================

EPOCHS_PHASE_2 = 20

total_steps = len(train_loader) * EPOCHS_PHASE_2

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)

print("Extended Phase 2 setup complete.")

Extended Phase 2 setup complete.


In [26]:
# ==========================================
# CONTINUE PHASE 2 TRAINING
# ==========================================
print("\nContinuing Phase 2 Fine-Tuning...")

for epoch in range(5, EPOCHS_PHASE_2):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        criterion,
        device,
        epoch,
        EPOCHS_PHASE_2
    )

    val_loss, val_acc, val_f1 = validate(
        model,
        val_loader,
        criterion,
        device,
        epoch,
        EPOCHS_PHASE_2
    )

    print(
        f"Phase 2 Epoch {epoch+1}: "
        f"Train Loss={train_loss:.4f}, "
        f"Train Acc={train_acc:.4f}, "
        f"Val Loss={val_loss:.4f}, "
        f"Val Acc={val_acc:.4f}, "
        f"Val F1={val_f1:.4f}"
    )

    wandb.log({
        'phase2_epoch': epoch + 1,
        'phase2_train_loss': train_loss,
        'phase2_train_accuracy': train_acc,
        'phase2_val_loss': val_loss,
        'phase2_val_accuracy': val_acc,
        'phase2_val_f1_weighted': val_f1,
    })

print("Extended Phase 2 Complete.")


Continuing Phase 2 Fine-Tuning...


Epoch 6/20 [Train]:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 6/20 [Val]:   0%|          | 0/15 [00:00<?, ?it/s]

Phase 2 Epoch 6: Train Loss=2.4007, Train Acc=0.2856, Val Loss=2.4195, Val Acc=0.2900, Val F1=0.2286


Epoch 7/20 [Train]:   0%|          | 0/59 [00:00<?, ?it/s]

...

Epoch 19/20 [Val]:   0%|          | 0/15 [00:00<?, ?it/s]

Phase 2 Epoch 19: Train Loss=1.9947, Train Acc=0.3865, Val Loss=2.1554, Val Acc=0.3220, Val F1=0.2732


Epoch 20/20 [Train]:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 20/20 [Val]:   0%|          | 0/15 [00:00<?, ?it/s]

Phase 2 Epoch 20: Train Loss=2.0108, Train Acc=0.3865, Val Loss=2.1674, Val Acc=0.3348, Val F1=0.2859
Extended Phase 2 Complete.


In [27]:
# ==========================================
# FINAL TEST EVALUATION
# ==========================================

test_loss, test_acc, test_f1 = validate(
    model,
    test_loader,
    criterion,
    device,
    0,
    1
)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score: {test_f1:.4f}")

Epoch 1/1 [Val]:   0%|          | 0/8 [00:00<?, ?it/s]


Test Loss: 2.1694
Test Accuracy: 0.3347
Test F1 Score: 0.2872


In [28]:
# ==========================================
# SAVE MODEL
# ==========================================

torch.save(
    model.state_dict(),
    "plantdoc_mobilenetv2.pth"
)

print("Model saved successfully.")

Model saved successfully.


In [29]:
from google.colab import files

files.download("plantdoc_mobilenetv2.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>